# Disease Classification - PlantVillage MobileNetV2 Fine-Tuning

This notebook trains a MobileNetV2-based classifier on the PlantVillage dataset
for crop disease detection. The trained model is exported as a quantized TFLite
model compatible with the Coral Edge TPU on the AgroBot rover.

**Hardware requirements:** Google Colab with **T4 GPU** or **v5e-1 TPU**
runtime — the accelerator-setup cell detects whichever is present (TPU gives
roughly 2-4x faster epochs). Note: the Cloud TPU only speeds up *training*;
the export still targets the rover's Coral **Edge** TPU, which is unrelated
hardware.

**Expected output:** `disease_model_quant_edgetpu.tflite`,
`disease_model_quant.tflite`, `disease_model_float16.tflite`, and
`plantvillage_labels.txt` for deployment to `models/` on the Raspberry Pi.

In [ ]:
# Environment setup.
# Colab's GPU/CPU runtimes already ship everything this notebook needs -
# pip-installing over the preinstalled stack breaks it, so on those runtimes
# this cell installs NOTHING.
# Only the TPU (v5e-1) runtime lacks TensorFlow: install the TPU build there.
import os

ON_TPU_VM = os.path.exists('/dev/accel0')  # present on Colab's v5e-1 TPU runtime
if ON_TPU_VM:
    %pip install -q tensorflow-tpu -f https://storage.googleapis.com/libtpu-tf-releases/index.html
    %pip install -q tf-keras
    os.environ['TF_USE_LEGACY_KERAS'] = '1'  # Keras 2: the proven TPUStrategy path

print('Setup OK -', 'TPU VM (installed TF-TPU)' if ON_TPU_VM else 'GPU/CPU (preinstalled stack, nothing installed)')


In [ ]:
import os

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")


In [ ]:
# Accelerator setup: TPU (Colab "v5e-1 TPU" runtime) when available, else GPU.
# TPU v5e needs TF >= 2.15 with PJRT - the tensorflow-tpu build installed
# above provides it. On a GPU runtime every attempt below fails harmlessly
# and training runs exactly as before under the default strategy.
strategy = None
for tpu_arg in ('local', None):
    try:
        resolver = (tf.distribute.cluster_resolver.TPUClusterResolver(tpu=tpu_arg)
                    if tpu_arg is not None
                    else tf.distribute.cluster_resolver.TPUClusterResolver())
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
        print(f"TPU ready: {strategy.num_replicas_in_sync} replica(s)")
        break
    except Exception:
        continue

ON_TPU = strategy is not None
if not ON_TPU:
    strategy = tf.distribute.get_strategy()
    print(f"No TPU; default strategy on {tf.config.list_physical_devices('GPU') or 'CPU'}")

In [ ]:
# Load PlantVillage straight from GitHub - 54,000+ images, 38 classes.
# tensorflow_datasets is NOT used: Colab's preinstalled copy is currently
# broken (protobuf gencode 6.31 vs runtime 5.29), so this only needs git + tf.
import pathlib
import shutil

REPO_DIR = pathlib.Path('PlantVillage-Dataset')
DATA_ROOT = REPO_DIR / 'raw' / 'color'

if REPO_DIR.exists() and not DATA_ROOT.exists():
    shutil.rmtree(REPO_DIR)  # broken partial download - start over
if not DATA_ROOT.exists():
    print('Downloading dataset (~2 GB, takes a few minutes)...')
    !git clone -q --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git
assert DATA_ROOT.exists(), 'Download failed - run this cell again'

CLASS_NAMES = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir())
NUM_CLASSES = len(CLASS_NAMES)
LABEL_OF = {name: i for i, name in enumerate(CLASS_NAMES)}

EXTS = {'.jpg', '.jpeg', '.png'}
all_paths = sorted(p for p in DATA_ROOT.glob('*/*') if p.suffix.lower() in EXTS)
all_labels = np.array([LABEL_OF[p.parent.name] for p in all_paths])

# Reproducible 80/10/10 train/val/test split
order = np.random.RandomState(42).permutation(len(all_paths))
n_train, n_val = int(0.8 * len(order)), int(0.1 * len(order))
SPLITS = {
    'train': order[:n_train],
    'val':   order[n_train:n_train + n_val],
    'test':  order[n_train + n_val:],
}


def make_dataset(indices):
    paths = [str(all_paths[i]) for i in indices]
    labels = all_labels[indices]

    def _load(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3,
                                   expand_animations=False)
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    return ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)


ds_train = make_dataset(SPLITS['train'])
ds_val = make_dataset(SPLITS['val'])
ds_test = make_dataset(SPLITS['test'])

assert NUM_CLASSES == 38, f'expected 38 classes, found {NUM_CLASSES}'
print(f"Number of classes: {NUM_CLASSES}")
print(f"Training samples: {len(SPLITS['train'])}")
print(f"Validation samples: {len(SPLITS['val'])}")
print(f"Test samples: {len(SPLITS['test'])}")


In [ ]:
# Data exploration and visualization (optional eye-candy - training does not
# depend on this cell). Robust to a batched/preprocessed dataset so an
# out-of-order re-run can't derail Run-all.
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, (image, label) in enumerate(ds_train.take(12)):
    img, lbl = image.numpy(), label.numpy()
    if img.ndim == 4:              # tolerate an already-batched dataset
        img, lbl = img[0], np.ravel(lbl)[0]
    ax = axes[i // 4, i % 4]
    ax.imshow(img.astype('uint8') if img.max() > 1.5 else img)
    ax.set_title(CLASS_NAMES[int(lbl)], fontsize=8)
    ax.axis('off')
plt.suptitle('PlantVillage Dataset Samples', fontsize=14)
plt.tight_layout()
plt.show()

# Class distribution - counted from the split label arrays (instant; no need
# to decode 43,000 images just to count them).
train_counts = np.bincount(all_labels[SPLITS['train']], minlength=NUM_CLASSES)
plt.figure(figsize=(14, 5))
plt.bar(range(NUM_CLASSES), train_counts)
plt.xlabel('Class Index')
plt.ylabel('Count')
plt.title('Class Distribution in Training Set')
plt.show()


In [ ]:
# Data augmentation pipeline
IMG_SIZE = 224
# Larger batches keep a TPU fed; 32 remains right for a T4.
BATCH_SIZE = 128 if ON_TPU else 32
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])


def preprocess_train(image, label):
    """Resize, normalize, and augment training images."""
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    image = data_augmentation(image, training=True)
    return image, label


def preprocess_eval(image, label):
    """Resize and normalize evaluation images (no augmentation)."""
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


# Build data pipelines. TPUs need static batch shapes -> drop_remainder.
train_ds = (
    ds_train
    .shuffle(1000)
    .map(preprocess_train, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=ON_TPU)
    .prefetch(AUTOTUNE)
)

val_ds = (
    ds_val
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=ON_TPU)
    .prefetch(AUTOTUNE)
)

test_ds = (
    ds_test
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=ON_TPU)
    .prefetch(AUTOTUNE)
)

print(f"Batch size: {BATCH_SIZE} ({'TPU' if ON_TPU else 'GPU/CPU'})")
print(f"Train batches: {len(train_ds)}")
print(f"Val batches: {len(val_ds)}")
print(f"Test batches: {len(test_ds)}")

In [ ]:
# Build MobileNetV2 transfer learning model.
# Built inside strategy.scope() so weights live on the TPU when one is active
# (a no-op on GPU/CPU - the default strategy scope changes nothing).
with strategy.scope():
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base_model.trainable = False  # freeze for feature extraction first

    # Custom classification head
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

model.summary()

In [ ]:
# Training with callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_disease_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
    ),
]

# Phase 1: Train with frozen base (feature extraction)
print("Phase 1: Feature extraction (frozen base)")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)

# Phase 2: Fine-tune top layers of base model
print("\nPhase 2: Fine-tuning top layers")
base_model.trainable = True
# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Re-compile inside the strategy scope (required on TPU).
with strategy.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
)

In [ ]:
# Evaluate with confusion matrix and classification report
# Load best model (inside the strategy scope so TPU/GPU placement matches)
with strategy.scope():
    model = tf.keras.models.load_model('best_disease_model.keras')

# Get predictions on test set
y_true = []
y_pred = []
for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Overall accuracy
accuracy = np.mean(y_true == y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Disease Classification')
plt.tight_layout()
plt.show()

# Training history plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_phase2.history['accuracy'], label='Train')
ax1.plot(history_phase2.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.legend()
ax2.plot(history_phase2.history['loss'], label='Train')
ax2.plot(history_phase2.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TFLite export - Float16 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_float16 = converter.convert()

os.makedirs('output', exist_ok=True)
with open('output/disease_model_float16.tflite', 'wb') as f:
    f.write(tflite_float16)
print(f"Float16 model size: {len(tflite_float16) / 1024 / 1024:.2f} MB")

# TFLite export - Full INT8 quantization (for Edge TPU)
def representative_dataset():
    """Generate representative dataset for INT8 calibration."""
    for images, _ in test_ds.take(100):
        for i in range(images.shape[0]):
            yield [tf.expand_dims(images[i], axis=0)]


converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8
tflite_int8 = converter_int8.convert()

with open('output/disease_model_quant.tflite', 'wb') as f:
    f.write(tflite_int8)
print(f"INT8 model size: {len(tflite_int8) / 1024 / 1024:.2f} MB")

In [ ]:
# Export the labels file - order MUST match the training label indices,
# so always regenerate it from CLASS_NAMES (alphabetical folder order), never by hand.
with open('output/plantvillage_labels.txt', 'w') as f:
    f.write('\n'.join(CLASS_NAMES) + '\n')
print(f"Wrote {len(CLASS_NAMES)} labels to output/plantvillage_labels.txt")

In [ ]:
# Edge TPU compilation (for the Coral on the rover)
# Install the Edge TPU compiler
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get update && sudo apt-get install -y edgetpu-compiler

# Compile the INT8 quantized model for Edge TPU
!edgetpu_compiler -s -o output/ output/disease_model_quant.tflite

print("\nCompiled models:")
!ls -la output/

In [ ]:
# Download everything for the Pi's models/ directory.
# pi/ai/disease_detection.py finds these filenames automatically - just copy
# all four files into models/ on the rover (or your repo):
#
#   models/disease_model_quant_edgetpu.tflite  -> Edge TPU model (Coral)
#   models/disease_model_quant.tflite          -> INT8 CPU model (no Coral)
#   models/disease_model_float16.tflite        -> CPU fallback model
#   models/plantvillage_labels.txt             -> class labels (training label order)
from google.colab import files

for name in ('disease_model_quant_edgetpu.tflite',
             'disease_model_quant.tflite',
             'disease_model_float16.tflite',
             'plantvillage_labels.txt'):
    path = f'output/{name}'
    if os.path.exists(path):
        files.download(path)
    else:
        print(f'MISSING: {path} (check the cell that should have produced it)')

print("Done! Copy the downloaded files into the models/ directory.")

## Optional: quantization-aware fine-tune (QAT)

Everything above this line is the normal pipeline and already produces working models — this section is an **optional add-on**.

The INT8 export above uses **post-training quantization (PTQ)**: weights and activations are quantized *after* training, which typically costs a few points of accuracy. **Quantization-aware training (QAT)** inserts fake-quantization nodes into the model and fine-tunes for a few epochs so the weights adapt to INT8 precision, usually recovering most of the PTQ loss.

**When to run:** after normal training and export. It starts from `best_disease_model.keras` on disk (the committed checkpoint) when present, otherwise from the in-session model — it never retrains from scratch. The dataset/pipeline cells above must have been run in this session (it reuses the same `train_ds`/`val_ds`/`test_ds` seed-42 80/10/10 split).

**Runtime:** use a **T4 GPU (or CPU)** runtime. `tensorflow_model_optimization` QAT is *not* reliably TPU-compatible; on the v5e-1 TPU runtime the cells below print a warning and fall back to the default strategy (very slow) — switch to T4 instead.

**Known limitation:** tfmot quantization targets the legacy Keras 2 implementation, and MobileNetV2 transfer models sometimes fail whole-model quantization. The build cell tries whole-model QAT first, falls back to quantizing only the Dense classifier head, and if both fail it prints instructions and leaves the PTQ export untouched.

**Output:** the export cell rewrites **the same file** `output/disease_model_quant.tflite`, so deployment filenames do not change; it also saves `best_disease_model_qat.keras` and prints PTQ-vs-QAT INT8 test accuracy side by side.

In [ ]:
# QAT step 1: install the toolkit. Kept in its own cell so an install failure
# cannot break anything above - the whole notebook works without this section.
%pip install -q tensorflow-model-optimization

In [ ]:
# QAT step 2: load the trained checkpoint and wrap it with fake-quant nodes.
# Needs train_ds/val_ds/test_ds from the pipeline cells above (re-run the
# setup + dataset cells first if this is a fresh session).
import tensorflow_model_optimization as tfmot

# tfmot QAT is not reliably TPU-compatible: force the default strategy there.
if ON_TPU:
    print('WARNING: TPU runtime detected. tensorflow_model_optimization QAT '
          'is not reliably TPU-compatible; building/fine-tuning under the '
          'default strategy instead (runs on the TPU VM CPU - slow). For '
          'real use, switch to a T4 GPU runtime and re-run the setup and '
          'dataset cells before this section.')
qat_strategy = tf.distribute.get_strategy() if ON_TPU else strategy

# Start from the best checkpoint on disk if present (it is also committed in
# the repo as training/best_disease_model.keras), else the in-session model.
if os.path.exists('best_disease_model.keras'):
    with qat_strategy.scope():
        float_model = tf.keras.models.load_model('best_disease_model.keras')
    print('Loaded best_disease_model.keras')
else:
    try:
        float_model = model
        print('best_disease_model.keras not found; using the in-session model')
    except NameError:
        raise FileNotFoundError(
            'No best_disease_model.keras on disk and no in-session model. '
            'Run the training cells first, or upload the checkpoint.')

qat_model = None
try:
    # Preferred path: whole-model QAT (quantizes MobileNetV2 base + head).
    with qat_strategy.scope():
        qat_model = tfmot.quantization.keras.quantize_model(float_model)
    print('Whole-model quantize_model() succeeded.')
except Exception as exc:
    print(f'Whole-model QAT failed: {type(exc).__name__}: {exc}')
    print('Falling back to quantizing only the Dense classifier head '
          '(common for MobileNetV2 transfer models / newer Keras versions).')
    try:
        annotate = tfmot.quantization.keras.quantize_annotate_layer
        with qat_strategy.scope():
            annotated = tf.keras.models.clone_model(
                float_model,
                clone_function=lambda layer: (
                    annotate(layer)
                    if isinstance(layer, tf.keras.layers.Dense) else layer
                ),
            )
            annotated.set_weights(float_model.get_weights())
            qat_model = tfmot.quantization.keras.quantize_apply(annotated)
        print('Head-only QAT model built.')
    except Exception as exc2:
        print(f'Head-only QAT also failed: {type(exc2).__name__}: {exc2}')
        print('This TF/Keras version is likely incompatible with tfmot '
              '(tfmot QAT requires the legacy Keras 2 implementation; on '
              'TF >= 2.16 set TF_USE_LEGACY_KERAS=1 before TensorFlow is '
              'first imported, then restart and re-run the notebook). '
              'Skipping QAT - the PTQ export above remains valid.')

if qat_model is not None:
    with qat_strategy.scope():
        qat_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )
    qat_model.summary()

In [ ]:
# QAT step 3: short fine-tune at a very low learning rate on the SAME
# train/val split built above. QAT is a refinement, not retraining - 3-5
# epochs is enough for the weights to adapt to the fake-quant noise.
if qat_model is None:
    print('QAT model unavailable - skipping the fine-tune.')
else:
    qat_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=2,
            restore_best_weights=True,
        ),
    ]
    qat_history = qat_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        callbacks=qat_callbacks,
    )
    # Separate filename on purpose: never clobber best_disease_model.keras.
    # (Re-loading this file later needs tfmot.quantization.keras.quantize_scope().)
    qat_model.save('best_disease_model_qat.keras')
    print('Saved best_disease_model_qat.keras')

In [ ]:
# QAT step 4: export INT8 TFLite from the QAT model and measure PTQ vs QAT.
# The PTQ file (if present) is evaluated FIRST, because the QAT export then
# overwrites THE SAME filename: output/disease_model_quant.tflite.
if qat_model is None:
    print('QAT model unavailable - keeping the PTQ export. Nothing changed.')
else:
    def eval_int8_tflite(tflite_path, dataset):
        """Accuracy of a uint8-in/uint8-out TFLite classifier on `dataset` -
        the TFLite-interpreter version of the Keras test-set eval above."""
        interpreter = tf.lite.Interpreter(model_path=tflite_path)
        interpreter.allocate_tensors()
        inp = interpreter.get_input_details()[0]
        out = interpreter.get_output_details()[0]
        scale, zero_point = inp['quantization']
        correct = total = 0
        for images, labels in dataset:
            images = images.numpy()
            labels = labels.numpy()
            for i in range(images.shape[0]):
                x = images[i:i + 1]
                if inp['dtype'] == np.uint8:
                    x = np.clip(np.round(x / scale + zero_point),
                                0, 255).astype(np.uint8)
                interpreter.set_tensor(inp['index'], x)
                interpreter.invoke()
                pred = interpreter.get_tensor(out['index'])[0]
                correct += int(np.argmax(pred) == int(labels[i]))
                total += 1
        return correct / total

    int8_path = 'output/disease_model_quant.tflite'
    ptq_acc = None
    if os.path.exists(int8_path):
        print('Evaluating the existing PTQ INT8 model on the test split '
              '(single-image TFLite CPU inference - takes a few minutes)...')
        ptq_acc = eval_int8_tflite(int8_path, test_ds)
        print(f'PTQ INT8 test accuracy: {ptq_acc:.4f}')
    else:
        print('No existing PTQ export found; run the TFLite export cell above '
              'first if you want the side-by-side comparison. Converting the '
              'QAT model anyway.')

    # SAME representative-dataset INT8 converter settings as the PTQ export
    # cell above - only the source model differs.
    def qat_representative_dataset():
        for images, _ in test_ds.take(100):
            for i in range(images.shape[0]):
                yield [tf.expand_dims(images[i], axis=0)]

    converter_qat = tf.lite.TFLiteConverter.from_keras_model(qat_model)
    converter_qat.optimizations = [tf.lite.Optimize.DEFAULT]
    converter_qat.representative_dataset = qat_representative_dataset
    converter_qat.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    ]
    converter_qat.inference_input_type = tf.uint8
    converter_qat.inference_output_type = tf.uint8
    tflite_qat_int8 = converter_qat.convert()

    os.makedirs('output', exist_ok=True)
    with open(int8_path, 'wb') as f:  # SAME deployment filename as PTQ
        f.write(tflite_qat_int8)
    print(f'QAT INT8 model written to {int8_path} '
          f'({len(tflite_qat_int8) / 1024 / 1024:.2f} MB)')

    print('Evaluating the QAT INT8 model on the test split...')
    qat_acc = eval_int8_tflite(int8_path, test_ds)
    print('\n=== INT8 test accuracy: PTQ vs QAT (same split, same converter) ===')
    print(f'  PTQ : {ptq_acc:.4f}' if ptq_acc is not None
          else '  PTQ : n/a (no prior export in this session)')
    print(f'  QAT : {qat_acc:.4f}')
    if ptq_acc is not None:
        print(f'  Delta: {qat_acc - ptq_acc:+.4f}')

### After the QAT export

- `output/disease_model_quant.tflite` now contains the **QAT** INT8 model under the exact filename the Pi expects (`pi/ai/disease_detection.py` finds models by filename), so zero Pi-side changes are needed.
- **Re-run the existing `edgetpu_compiler` cell above** (do not duplicate it) so `output/disease_model_quant_edgetpu.tflite` is rebuilt from the QAT model, then re-run the download cell.
- `best_disease_model_qat.keras` holds the fine-tuned QAT Keras model. Loading it later requires wrapping the load in `tfmot.quantization.keras.quantize_scope()`.
- If the QAT accuracy printed above came out *lower* than PTQ (possible when only the head could be quantized, or when the fine-tune diverged), re-run the original TFLite export cell to restore the PTQ file — the filenames are identical either way.